In [1]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
import requests


os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
)

/home/devsafix/Projects/Learning/langchain-genai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Built in tools

In [3]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('What is weather of the Dhaka today')

print(results)

May 2, 2026 - Get Dhaka, C, BD current weather report with temperature, feels like, wind, humidity, pressure, UV and more from TheWeatherNetwork.com. 4 days ago - ... The weather in Dhaka today is expected to be slightly warmer than usual, with a forecast temperature of 36 °C, compared to an average of 33.7 °C for 18th of June in recent years. 3 weeks ago - Weather in Dhaka right now: clear, air temperature +30°, feels like +34°. Wind speed and direction is 1.7 m/s, SW, humidity is 68%, atmospheric pressure is 753 mmHg. No precipitation is expected for the next 2 hours. Today: +28…+30°, clear, no precipitation, light winds at 1 m/s. 4 weeks ago - ... No active precipitation in Dhaka right now. Use the hourly forecast above to see chances of rain through the day. ... Today in Dhaka, temperatures range from a low of 27°C to a high of 36°C, currently 36°C. 1 week ago - The current temperature in Dhaka is 32°C, with a feels-like temperature of 37°C. Today's forecast shows a high of 35°C an

In [4]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


### Custom Tools

In [5]:
@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

result = multiply.invoke({"a":3, "b":5})

print(result)
print(multiply.name)
print(multiply.description)
print(multiply.args)

15
multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


### Using StructuredTool

In [9]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class MultiplyInput(BaseModel):
    a: int = Field(description="The first number to add")
    b: int = Field(description="The second number to add")

def multiply_func(a: int, b: int) -> int:
    return a * b

multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


### Using baseTool class

In [12]:
from langchain.tools import BaseTool
from typing import Type

class MultiplyInput(BaseModel):
    a: int = Field(description="The first number to add")
    b: int = Field(description="The second number to add")

class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

multiply_tool = MultiplyTool()

result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}
